# BERT SHAP 해석

KLUE-RoBERTa 최종 모델의 예측 근거를 토큰 수준에서 분석합니다.

| 분석 대상 | 케이스 | 예상 시간 |
|-----------|--------|-----------|
| 이진 분류 | TP/TN/FP/FN 각 3건 | ~10분 |
| 다중 분류 | 유형별 1건 + 혼동 2건 | ~8분 |

> ⚠️ SHAP은 1건당 30~60초 소요. 셀 실행 후 기다리세요.

## Step 0. 패키지 설치

In [ ]:
!pip install -q transformers torch sentencepiece protobuf scikit-learn tqdm shap

## Step 1. Google Drive 마운트 + 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/text-mining-2026/data/processed'
SAVE_DIR = '/content/drive/MyDrive/text-mining-2026/models'

print(f'데이터: {DATA_DIR}')
print(f'모델:   {SAVE_DIR}')

## Step 2. 라이브러리 임포트

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, pipeline
)
import shap

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 3. test_final 로딩 + 설정값

In [ ]:
df_test = pd.read_parquet(f'{DATA_DIR}/test_final.parquet')
print(f'test_final: {len(df_test):,}건')

MODEL_NAME = 'klue/roberta-base'
MAX_LENGTH = 512
BATCH_SIZE = 64
CONTENT_LEN = 200  # SHAP 입력용 본문 길이 제한

TYPE_NAMES = {
    0: '의문유발-부호', 1: '의문유발-은닉', 2: '선정표현',
    3: '속어/줄임말',  4: '사실과대',      5: '주어왜곡',
}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('토크나이저 로드 완료')

## Step 4. Dataset + 배치 추론 함수

In [ ]:
class ClickbaitDataset(Dataset):
    def __init__(self, dataframe, tokenizer, task='binary'):
        self.titles   = dataframe['title_clean'].tolist()
        self.contents = dataframe['content_clean'].tolist()
        self.labels   = dataframe[
            'binary_label' if task == 'binary' else 'type_label'
        ].tolist()
        self.tok = tokenizer

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tok(
            self.titles[idx], self.contents[idx],
            truncation='only_second', max_length=MAX_LENGTH,
            padding='max_length', return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }


def batch_predict(model, df, task='binary'):
    """전체 df에 대해 배치 추론 → 결과 컬럼(pred, true) 추가한 df 반환"""
    if task == 'multi':
        df = df[df['type_label'] != -1].reset_index(drop=True)

    dataset = ClickbaitDataset(df, tokenizer, task=task)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2, pin_memory=True)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f'[{task}] 배치 추론'):
            out = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
            )
            all_preds.extend(torch.argmax(out.logits, dim=-1).cpu().numpy())
            all_labels.extend(batch['labels'].numpy())

    df = df.copy()
    df['pred'] = all_preds
    df['true'] = all_labels
    return df


def make_shap_text(row):
    """제목 + 본문 앞부분 연결 (SHAP 입력용)"""
    return row['title_clean'] + ' ' + row['content_clean'][:CONTENT_LEN]


def show_shap_table(shap_val, top_n=20, task='binary'):
    """
    shap.plots.text() 대신 상위 기여 토큰을 표로 출력.
    - 양수(+): 해당 클래스 확률을 높이는 토큰 → 빨간색
    - 음수(-): 해당 클래스 확률을 낮추는 토큰 → 파란색
    """
    from IPython.display import display

    tokens = list(shap_val.data)
    values = shap_val.values  # (n_tokens, n_labels)

    def _color(val):
        if not isinstance(val, float):
            return ''
        alpha = min(0.85, abs(val) * 12)
        if val > 0:
            return f'background-color: rgba(220, 50, 50, {alpha:.2f}); color: white'
        else:
            return f'background-color: rgba(50, 100, 220, {alpha:.2f}); color: white'

    if task == 'binary':
        # LABEL_1(낚시성) 기여도 기준으로 정렬
        scores = values[:, 1]
        df_s = pd.DataFrame({'토큰': tokens, '낚시성 기여도 (LABEL_1)': scores})
        df_s['abs'] = df_s['낚시성 기여도 (LABEL_1)'].abs()
        df_s = df_s.nlargest(top_n, 'abs').drop(columns='abs').reset_index(drop=True)

        print(f'  ▶ 예측 확률: 정상={shap_val.base_values[0]+values[:,0].sum():.4f} | '
              f'낚시성={shap_val.base_values[1]+values[:,1].sum():.4f}')
        display(
            df_s.style
            .applymap(_color, subset=['낚시성 기여도 (LABEL_1)'])
            .format({'낚시성 기여도 (LABEL_1)': '{:+.5f}'})
            .set_caption('빨강=낚시성 기여 / 파랑=정상 기여 (상위 토큰)')
        )

    else:  # multi
        label_cols = [TYPE_NAMES[i] for i in range(len(TYPE_NAMES))]
        df_s = pd.DataFrame({'토큰': tokens})
        for i, col in enumerate(label_cols):
            df_s[col] = values[:, i]
        df_s['max_abs'] = np.abs(values).max(axis=1)
        df_s = df_s.nlargest(top_n, 'max_abs').drop(columns='max_abs').reset_index(drop=True)

        display(
            df_s.style
            .applymap(_color, subset=label_cols)
            .format({col: '{:+.5f}' for col in label_cols})
            .set_caption('빨강=해당 유형 기여 / 파랑=해당 유형 억제 (상위 토큰)')
        )


print('함수 정의 완료')

## Step 5. [이진] 전체 추론 → TP/TN/FP/FN 분류

In [ ]:
model_bin = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model_bin.load_state_dict(
    torch.load(f'{SAVE_DIR}/klue_binary_final.pt', map_location=device)
)
model_bin.to(device)

df_bin = batch_predict(model_bin, df_test, task='binary')

TP = df_bin[(df_bin['true']==1) & (df_bin['pred']==1)].reset_index(drop=True)
TN = df_bin[(df_bin['true']==0) & (df_bin['pred']==0)].reset_index(drop=True)
FP = df_bin[(df_bin['true']==0) & (df_bin['pred']==1)].reset_index(drop=True)
FN = df_bin[(df_bin['true']==1) & (df_bin['pred']==0)].reset_index(drop=True)

print(f'TP (낚시→낚시 정답):  {len(TP):,}건')
print(f'TN (정상→정상 정답):  {len(TN):,}건')
print(f'FP (정상→낚시 오분류): {len(FP):,}건')
print(f'FN (낚시→정상 오분류): {len(FN):,}건')

## Step 6. [이진] SHAP 분석

> ⚠️ 약 10~15분 소요. 빨간 토큰 = 낚시성 기여, 파란 토큰 = 정상 기여

In [ ]:
N = 3  # 카테고리당 샘플 수

pipe_bin = pipeline(
    'text-classification',
    model=model_bin,
    tokenizer=tokenizer,
    top_k=None,
    device=0 if torch.cuda.is_available() else -1,
)
explainer_bin = shap.Explainer(pipe_bin)

binary_cases = [
    ('TP — 낚시성 정답 (낚시→낚시)', TP.head(N)),
    ('TN — 정상 정답 (정상→정상)',   TN.head(N)),
    ('FP — 오분류: 정상→낚시',       FP.head(N)),
    ('FN — 오분류: 낚시→정상',       FN.head(N)),
]

for case_name, df_case in binary_cases:
    if len(df_case) == 0:
        print(f'\n=== {case_name} — 케이스 없음 ===')
        continue
    print(f'\n{"="*60}')
    print(f'  {case_name}')
    print(f'{"="*60}')

    texts = [make_shap_text(row) for _, row in df_case.iterrows()]
    shap_vals = explainer_bin(texts)

    for i, (_, row) in enumerate(df_case.iterrows()):
        label_map = {0: '정상', 1: '낚시성'}
        print(f'\n[{i+1}] 제목: {row["title_clean"]}')
        print(f'     정답={label_map[int(row["true"])]} | 예측={label_map[int(row["pred"])]}')
        show_shap_table(shap_vals[i], top_n=20, task='binary')

## Step 7. [다중] 전체 추론 → 유형별 / 혼동 케이스 분류

In [ ]:
del model_bin
torch.cuda.empty_cache()

model_multi = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=6)
model_multi.load_state_dict(
    torch.load(f'{SAVE_DIR}/klue_multi_final.pt', map_location=device)
)
model_multi.to(device)

df_multi = batch_predict(model_multi, df_test, task='multi')

print('유형별 추론 결과:')
for lbl, name in TYPE_NAMES.items():
    correct = df_multi[(df_multi['true']==lbl) & (df_multi['pred']==lbl)]
    wrong   = df_multi[(df_multi['true']==lbl) & (df_multi['pred']!=lbl)]
    print(f'  {name:12s}: 정답 {len(correct):,}건 | 오분류 {len(wrong):,}건')

## Step 8. [다중] SHAP 분석

> 유형별 정답 1건 + 의문유발-부호↔은닉 혼동 2건
> ⚠️ 약 8~12분 소요

In [ ]:
pipe_multi = pipeline(
    'text-classification',
    model=model_multi,
    tokenizer=tokenizer,
    top_k=None,
    device=0 if torch.cuda.is_available() else -1,
)
explainer_multi = shap.Explainer(pipe_multi)

multi_cases = []
for lbl in range(6):
    correct = df_multi[(df_multi['true']==lbl) & (df_multi['pred']==lbl)]
    if len(correct) > 0:
        multi_cases.append((f'{TYPE_NAMES[lbl]} — 정답', correct.head(1)))

confused_01 = df_multi[(df_multi['true']==0) & (df_multi['pred']==1)]
confused_10 = df_multi[(df_multi['true']==1) & (df_multi['pred']==0)]
if len(confused_01) > 0:
    multi_cases.append(('부호→은닉 오분류 (부호인데 은닉으로 예측)', confused_01.head(2)))
if len(confused_10) > 0:
    multi_cases.append(('은닉→부호 오분류 (은닉인데 부호로 예측)', confused_10.head(2)))

for case_name, df_case in multi_cases:
    if len(df_case) == 0:
        continue
    print(f'\n{"="*60}')
    print(f'  {case_name}')
    print(f'{"="*60}')

    texts = [make_shap_text(row) for _, row in df_case.iterrows()]
    shap_vals = explainer_multi(texts)

    for i, (_, row) in enumerate(df_case.iterrows()):
        true_name = TYPE_NAMES[int(row['true'])]
        pred_name = TYPE_NAMES[int(row['pred'])]
        print(f'\n[{i+1}] 제목: {row["title_clean"]}')
        print(f'     정답={true_name} | 예측={pred_name}')
        show_shap_table(shap_vals[i], top_n=20, task='multi')

## Step 9. 해석 포인트 정리

SHAP 결과를 볼 때 확인할 것:

| 분석 | 확인 포인트 |
|------|------------|
| TP (낚시 정답) | 낚시 판단에 기여한 토큰이 실제 낚시 표현인가? (충격, 경악 등) |
| FP (정상→낚시) | 모델이 왜 낚시로 봤는가? 어떤 토큰이 오판 유발? |
| FN (낚시→정상) | 모델이 놓친 낚시 패턴은 무엇인가? |
| 유형별 정답 | 각 유형의 핵심 토큰이 TF-IDF 키워드 분석과 일치하는가? |
| 부호↔은닉 혼동 | 부호(?) 없을 때 모델이 어떤 다른 토큰에 집중하는가? |